# 05 - Model Final Task-1

Notebook ini menjalankan pipeline training LightGBM yang sudah diperbaiki:

- load data fitur hasil preprocessing
- target encoding out-of-fold
- count encoding
- validasi Macro F1
- tuning bobot kelas
- training model final
- penyimpanan model dan `submission.csv`

Jalankan seluruh cell dari atas ke bawah.

## Persiapan
Pastikan environment memiliki `pandas`, `numpy`, `lightgbm`, `scikit-learn`, dan `pyarrow`.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd()
while not (ROOT / "pitri").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

os.chdir(ROOT)

# 50.000 baris untuk tes cepat:
# os.environ["SUB_N"] = "50000"
# os.environ["ROUNDS"] = "120"

# Training final:
os.environ["SUB_N"] = "1050000"
os.environ["ROUNDS"] = "800"
os.environ["LR"] = "0.05"

print("Project root:", ROOT)

In [ ]:
import runpy

runpy.run_path(
    "pitri/src/train_v2.py",
    run_name="__main__"
)

In [ ]:
import pandas as pd

submission = pd.read_csv(ROOT / "pitri" / "results" / "submission.csv")

print("Bentuk submission:", submission.shape)
print("Ada nilai kosong:", submission.isna().sum().to_dict())
print("Jumlah shipment unik:", submission["shipment_id"].nunique())
print("\nDistribusi prediksi:")
print(submission["status_penanganan"].value_counts())

classes = set(["normal", "konfirmasi_alamat", "tertunda_operasional",
               "gagal_antar_awal", "retur_pengirim", "rusak_perjalanan",
               "klaim_hilang"])

assert submission.shape == (362245, 2)
assert submission["shipment_id"].is_unique
assert submission["status_penanganan"].notna().all()
assert set(submission["status_penanganan"]).issubset(classes)

print("\nValidasi submission: LULUS")
submission.head()